# AeroFleet — 1,000-case mass forensics run on Colab (free GPU)

Colab-side counterpart to `research/kaggle_mass_forensics_run.ipynb`. Same harness, same real
Ollama backend, same model substitution, same non-blocking background-process pattern so you can
check progress without stopping the run. Runs a **different, non-overlapping batch range** so it
works in parallel with the Kaggle run (and a third machine, if you have one).

## Required one-time manual setup (Colab UI, not this notebook)

1. **Runtime → Change runtime type → T4 GPU**, then Save (this reconnects the runtime).
2. **Secrets** (key icon in the left sidebar) → **Add new secret** → name it `GH_PAT`, value =
   a GitHub fine-grained token, read-only, scoped to just `AdityaPathare46/aerofleet` (private
   repo — reuse the same token you made for Kaggle). Toggle **Notebook access** on.
3. This notebook mounts your Google Drive and writes results there directly — progress survives a
   disconnect automatically, no manual save step needed. First time only: **File → Save a copy in
   Drive** so you can reopen this notebook tomorrow without re-uploading it.

## Which batch range this notebook runs

`--target 1000 --batch-size 25` gives 40 batches total.

| Machine | Batches | Cases |
|---|---|---|
| Kaggle | 1-14 | MFI-00001 – MFI-00350 |
| **This Colab notebook** | **27-40** | **MFI-00651 – MFI-01000** |
| Third machine | 15-26 | MFI-00351 – MFI-00650 |

Change `ONLY_BATCHES` in the run cell if you want a different split — just keep all machines'
ranges disjoint and covering 1-40 together.

## Same model-substitution caveat as the Kaggle run

`llama4:scout` (67GB) doesn't fit a free GPU. The 4 agents it powers (DISPATCHER,
AIRSPACE_SAFETY, AI_VALIDATOR, CONTINGENCY) use `mistral-nemo:12b` instead here too — **this must
match exactly what the Kaggle and third-machine runs use**, or the pooled results mix different
"real" configurations into one dataset without that being traceable per-case.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 1. Mount Drive (for persistence) and clone the private repo


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from google.colab import userdata
_token = userdata.get('GH_PAT')

REPO_DIR = "/content/aerofleet"
import os
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://{_token}@github.com/AdityaPathare46/aerofleet.git {REPO_DIR}
else:
    print("Repo already present — pulling latest instead of a fresh clone.")
    !cd {REPO_DIR} && git pull

del _token
%cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt


## 2. Install Ollama and start the server in this session

`OLLAMA_KEEP_ALIVE=-1` tells Ollama to never voluntarily unload a model due to idle time — one of
two possible causes (the other being genuine VRAM capacity) of a model getting evicted and
reloaded between agent calls, which shows up as a multi-minute gap in the logs for no obvious
reason. Costs nothing to set; only helps if idle-eviction was part of the problem.


In [ ]:
# zstd: required by the Ollama installer to extract its archive.
# pciutils (lspci): lets the installer auto-detect the GPU and install the
# matching CUDA runtime — without it, install silently warns and may fall
# back to a CPU-only build, which would make the run far too slow to finish.
!apt-get update -qq && apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time, requests, os

os.environ["OLLAMA_KEEP_ALIVE"] = "-1"

log = open("/content/ollama_serve.log", "a")
ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy())

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=5)
        print("Ollama server is up.")
        break
    except requests.exceptions.RequestException:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not come up — check /content/ollama_serve.log")

time.sleep(2)
!grep -i -E "gpu|cuda|library=" /content/ollama_serve.log | tail -5


## 3. Pull the models

Same 4 models as the Kaggle run (~41GB total). Re-run this cell if a pull is interrupted —
Ollama resumes partial downloads.


In [ ]:
for model in ["gemma4:12b", "phi4-reasoning:plus", "mistral-small3.2", "mistral-nemo:12b"]:
    print(f"--- pulling {model} ---")
    !ollama pull {model}

!ollama list


## 4. Study directory on Drive (persists automatically) + this machine's batch range

No resume step needed like Kaggle — since `STUDY_DIR` lives on Drive, a previous session's
`results.jsonl` is just already there.


In [ ]:
from pathlib import Path

STUDY_DIR = Path("/content/drive/MyDrive/aerofleet_mass_forensics_colab")
STUDY_DIR.mkdir(parents=True, exist_ok=True)

n_prior = sum(1 for _ in open(STUDY_DIR / "results.jsonl")) if (STUDY_DIR / "results.jsonl").exists() else 0
print(f"{n_prior} case-attempts already on Drive from a previous session (0 is normal for a first run).")


## 5. Configure the model substitution and record it (must match the other machines)


In [ ]:
import os, json, subprocess, datetime

os.environ["OLLAMA_HOST"] = "http://localhost:11434"
os.environ.pop("USE_MOCK_AGENTS", None)

for agent_id in ["DISPATCHER", "AIRSPACE_SAFETY", "AI_VALIDATOR", "CONTINGENCY"]:
    os.environ[f"AGENT_MODEL_{agent_id}"] = "mistral-nemo:12b"

gpu_name = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
).stdout.strip()

metadata = {
    "platform": "colab",
    "gpu": gpu_name,
    "only_batches": "27-40",
    "run_started_utc": datetime.datetime.utcnow().isoformat(),
    "substitution": {
        "replaced_model": "llama4:scout",
        "substitute_model": "mistral-nemo:12b",
        "reason": "llama4:scout is 67GB (109B-param MoE); does not fit a free-tier GPU",
        "affected_agents": ["DISPATCHER", "AIRSPACE_SAFETY", "AI_VALIDATOR", "CONTINGENCY"],
    },
}
with open(STUDY_DIR / "cloud_run_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))


## 6. Start the harness — runs in the background

This does NOT block the notebook — it starts the run and returns immediately. Use the cells below
to check on it, any time, as often as you like, without needing to stop it first.


In [ ]:
import subprocess, os

ONLY_BATCHES = "27-40"  # this machine's assigned, non-overlapping slice — see the table above.

harness_log_path = STUDY_DIR / "harness_run.log"
harness_log = open(harness_log_path, "a")

cmd = [
    "python", "-m", "scenario_engine.mass_forensics_evaluation",
    "--study-dir", str(STUDY_DIR), "--target", "1000", "--batch-size", "25",
]
if ONLY_BATCHES:
    cmd += ["--only-batches", ONLY_BATCHES]

harness_proc = subprocess.Popen(cmd, stdout=harness_log, stderr=subprocess.STDOUT, env=os.environ.copy())
print(f"Harness started in the background, PID {harness_proc.pid}.")
print(f"Logging to {harness_log_path}")
print("Run the cells below any time to check progress — no need to wait or stop this first.")


## 7. Monitoring — run any of these any time, as often as you like


In [ ]:
# Tail the live log
!tail -n 40 {harness_log_path}


In [ ]:
# Is it still running? (None = still running; a number = exit code, it finished/stopped)
print("exit code (None = still running):", harness_proc.poll())


In [ ]:
# How many case-attempts recorded so far
results_file = STUDY_DIR / "results.jsonl"
if results_file.exists():
    with open(results_file) as f:
        n = sum(1 for _ in f)
    print(f"{n} case-attempts recorded so far in this machine's range (27-40).")
else:
    print("No results yet.")


In [ ]:
# Thrashing check: if this shows a model, then a few cells later shows a DIFFERENT model
# (or nothing) repeatedly, models are being evicted and reloaded between calls — worth flagging.
!ollama ps


## 8. Stopping cleanly

Use this instead of Colab's "interrupt execution" — it terminates only the harness process,
leaving the Ollama server running undisturbed (an interrupt at the notebook level may not make
that distinction).


In [ ]:
harness_proc.terminate()
harness_proc.wait(timeout=30)
print("Harness stopped cleanly. results.jsonl on Drive has everything completed up to this point.")


## 9. Resuming tomorrow

Reopen this notebook from Drive, Runtime → reconnect, reselect T4 GPU, and re-run every cell from
the top (the Colab machine itself is wiped — repo clone, Ollama install, and model pull all
happen again). No manual resume step needed: since `STUDY_DIR` already points at the same Drive
folder, the harness sees yesterday's `results.jsonl` there automatically and just continues.

## 10. Merging all three machines' results

Once Kaggle, this Colab notebook, and the third machine have each made progress (they don't all
need to be *finished* — merging is safe at any point), pull all three `results.jsonl` +
`dataset_manifest.json` + `run_config.json` sets down to one place and run:

```bash
python -m research.merge_batched_results \
    --source /path/to/kaggle_study \
    --source /path/to/colab_study \
    --source /path/to/friend_study \
    --output /path/to/merged_study

python -m scenario_engine.mass_forensics_evaluation --study-dir /path/to/merged_study --report-only
```

The merge script refuses to combine studies that weren't generated with the same
`--target`/`--batch-size`/`--seed` (they wouldn't share a manifest), and warns if the same
case_id shows up from two sources — a sign the batch ranges weren't actually disjoint.
